In [1]:
import torch
import torch.nn.functional as F   
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from safetensors.torch import load_file

device = 'cuda:0'



In [2]:

basemodel_id = "CompVis/stable-diffusion-v1-4"
torch_dtype = torch.bfloat16

unet_0 = UNet2DConditionModel.from_pretrained(basemodel_id, subfolder="unet").to(device, torch_dtype)
unet_u = UNet2DConditionModel.from_pretrained(basemodel_id, subfolder="unet").to(device, torch_dtype)

unet_0.requires_grad_(False)

pipe = StableDiffusionPipeline.from_pretrained(basemodel_id, unet=unet_0, torch_dtype=torch_dtype).to(device)



path = './data_root/logs/esd/study/esd-x-kv.bG.fG.T999-1000_U.obama_sd1.4.bf16.bs4_AL0.00-0.00_r0/step500.safetensors'
unet_u.load_state_dict(load_file(path), strict=False)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/home/nessessence/anaconda3/envs/uul/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


_IncompatibleKeys(missing_keys=['conv_in.weight', 'conv_in.bias', 'time_embedding.linear_1.weight', 'time_embedding.linear_1.bias', 'time_embedding.linear_2.weight', 'time_embedding.linear_2.bias', 'down_blocks.0.attentions.0.norm.weight', 'down_blocks.0.attentions.0.norm.bias', 'down_blocks.0.attentions.0.proj_in.weight', 'down_blocks.0.attentions.0.proj_in.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.0.bias', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.weight', 'down_blocks.0.attentions.0.transformer_blocks.0.norm2.bias', 'down_blocks.0.attentions.0.t

In [3]:
def compute_angular_exclusion_inclusion_loss(
    unet_u,         # unlearned UNet (trainable)
    unet_0,         # frozen reference UNet
    p_e,            # erased prompt embedding [B, T, 768]
    p_g,            # generic prompt embedding [B, T, 768]
    m_excl: float,
    m_incl: float,
    layer_filter="attn2",  # cross-attention in diffusers SD1.4
    use_bias: bool = False,
    sim_param_group: str = "avg_token",  # {'avg_token', 'token', 'attn_head'}
):
    """
    Angular exclusion + inclusion loss in KV projection space (unsquared hinge).

    sim_param_group:
      - 'avg_token' : mean over tokens -> cosine -> hinge
      - 'token'     : cosine per token -> hinge per token -> mean
      - 'attn_head' : mean over tokens -> reshape to (num_heads=8) -> cosine per head
                     -> hinge per head -> mean

    Also returns L_norm (NOT added to L_ang):
      L_norm = mean | log(||W_u p_e||) - log(||W_0 p_e||) |
    computed using the same grouping as sim_param_group.

    Returns:
      (L_excl, L_incl, L_ang, L_norm)
    """

    assert sim_param_group in {"avg_token", "token", "attn_head"}

    params_u = dict(unet_u.named_parameters())
    params_0 = dict(unet_0.named_parameters())

    excl_terms = []
    incl_terms = []
    norm_terms = []
    matched_layers = 0

    for name, W_u in params_u.items():
        if layer_filter not in name:
            continue
        if not (name.endswith("to_k.weight") or name.endswith("to_v.weight")):
            continue
        if name not in params_0:
            continue

        # Frozen reference
        W_0 = params_0[name].detach()

        # Bias handling
        if use_bias:
            b_name = name.replace(".weight", ".bias")
            b_u = params_u.get(b_name, None)
            b_0 = params_0.get(b_name, None)
            if b_0 is not None:
                b_0 = b_0.detach()
        else:
            b_u = None
            b_0 = None

        # Linear projections: [B, T, D]
        W_u_e = F.linear(p_e, W_u, b_u)
        with torch.no_grad():
            W_0_e = F.linear(p_e, W_0, b_0)
            W_0_g = F.linear(p_g, W_0, b_0)

        if sim_param_group == "avg_token":
            W_u_e_m = W_u_e.mean(dim=1)
            W_0_e_m = W_0_e.mean(dim=1)
            W_0_g_m = W_0_g.mean(dim=1)

            cos_excl = F.cosine_similarity(W_u_e_m, W_0_e_m, dim=-1)
            cos_incl = F.cosine_similarity(W_u_e_m, W_0_g_m, dim=-1)

            excl_terms.append(torch.clamp(cos_excl - m_excl, min=0.0).mean())
            incl_terms.append(torch.clamp(m_incl - cos_incl, min=0.0).mean())

            norm_u = W_u_e_m.norm(dim=-1)
            norm_0 = W_0_e_m.norm(dim=-1)
            norm_terms.append(torch.abs(torch.log(norm_u) - torch.log(norm_0)).mean())

        elif sim_param_group == "token":
            cos_excl_tok = F.cosine_similarity(W_u_e, W_0_e, dim=-1)
            cos_incl_tok = F.cosine_similarity(W_u_e, W_0_g, dim=-1)

            excl_terms.append(torch.clamp(cos_excl_tok - m_excl, min=0.0).mean())
            incl_terms.append(torch.clamp(m_incl - cos_incl_tok, min=0.0).mean())

            norm_u = W_u_e.norm(dim=-1)
            norm_0 = W_0_e.norm(dim=-1)
            norm_terms.append(torch.abs(torch.log(norm_u) - torch.log(norm_0)).mean())

        elif sim_param_group == "attn_head":
            W_u_e_m = W_u_e.mean(dim=1)
            W_0_e_m = W_0_e.mean(dim=1)
            W_0_g_m = W_0_g.mean(dim=1)

            B, D = W_u_e_m.shape
            num_heads = 8
            assert D % num_heads == 0
            head_dim = D // num_heads

            W_u_e_h = W_u_e_m.view(B, num_heads, head_dim)
            W_0_e_h = W_0_e_m.view(B, num_heads, head_dim)
            W_0_g_h = W_0_g_m.view(B, num_heads, head_dim)

            cos_excl_h = F.cosine_similarity(W_u_e_h, W_0_e_h, dim=-1)
            cos_incl_h = F.cosine_similarity(W_u_e_h, W_0_g_h, dim=-1)

            excl_terms.append(torch.clamp(cos_excl_h - m_excl, min=0.0).mean())
            incl_terms.append(torch.clamp(m_incl - cos_incl_h, min=0.0).mean())

            norm_u = W_u_e_h.norm(dim=-1)
            norm_0 = W_0_e_h.norm(dim=-1)
            norm_terms.append(torch.abs(torch.log(norm_u) - torch.log(norm_0)).mean())

        matched_layers += 1

    if matched_layers == 0:
        zero = torch.tensor(0.0, device=p_e.device)
        return zero, zero, zero, zero

    L_excl = torch.stack(excl_terms).mean()
    L_incl = torch.stack(incl_terms).mean()
    L_ang = L_excl + L_incl
    L_norm = torch.stack(norm_terms).mean()

    return L_excl, L_incl, L_norm, L_ang

In [4]:
prompt = ['a photo of Barack Obama','a photo of person','a photo of Morgan Freeman', 'a photo of Will Smith','a photo of cat']
# prompt = ['Barack Obama','person','a photo of Morgan Freeman', 'a photo of Will Smith','a photo of cat']
with torch.no_grad():
    text_embedding, _ = pipe.encode_prompt(prompt=prompt,
                                            device=device,
                                            num_images_per_prompt=1,
                                            do_classifier_free_guidance=False)  
    p_e, p_g,p_p0, p_p1, p_o0 = text_embedding.chunk(5, dim=0)

In [6]:
compute_angular_exclusion_inclusion_loss(
    unet_u=unet_u,
    unet_0=unet_0,
    p_e=p_e,
    p_g=p_g,
    m_excl=0.0,
    m_incl=0.0,
    layer_filter="attn2",
    # sim_param_group="avg_token"
    
    sim_param_group="attn_head"
)

(tensor(0.2637, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.0791, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.2637, device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>))

In [7]:
compute_angular_exclusion_inclusion_loss(
    unet_u=unet_u,
    unet_0=unet_0,
    p_e=p_e,
    p_g=p_p0,
    m_excl=0.0,
    m_incl=0.0,
    layer_filter="attn2",
    # sim_param_group="avg_token"
    
    sim_param_group="attn_head"
)

(tensor(0.2637, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.0010, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.0791, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0.2656, device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>))

In [17]:
compute_angular_exclusion_inclusion_loss(
    unet_u=pipe.unet,
    unet_0=pipe.unet,
    p_e=p_e,
    p_g=p_p0,
    m_excl=0.0,
    m_incl=0.0,
    layer_filter="attn2",
    # sim_param_group="avg_token"
    
    sim_param_group="attn_head"
)

(tensor(1., device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(1., device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>))

In [73]:
compute_angular_exclusion_inclusion_loss(
    unet_u=pipe.unet,
    unet_0=pipe.unet,
    p_e=p_e,
    p_g=p_p1,
    m_excl=0.0,
    m_incl=0.0,
    layer_filter="attn2",
    # sim_param_group="avg_token"
    
    sim_param_group="attn_head"
)

tensor([[0.8047, 0.6602, 0.8203, 0.8125, 0.7031, 0.7344, 0.8555, 0.9219]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.8281, 0.9102, 0.7812, 0.9219, 0.7852, 0.8789, 0.8711, 0.9180]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.8398, 0.7852, 0.8281, 0.7734, 0.9062, 0.8789, 0.9180, 0.8203]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.9141, 0.8867, 0.9102, 0.9023, 0.9062, 0.9336, 0.9531, 0.9180]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.6367, 0.7578, 0.6367, 0.6406, 0.7344, 0.7539, 0.7070, 0.5820]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.8359, 0.9336, 0.7266, 0.6641, 0.8164, 0.8047, 0.7266, 0.6797]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.7695, 0.8086, 0.7422, 0.8242, 0.7930, 0.8281, 0.7891, 0.8438]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<

(tensor(1., device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(1., device='cuda:3', dtype=torch.bfloat16, grad_fn=<AddBackward0>))

In [74]:
compute_angular_exclusion_inclusion_loss(
    unet_u=pipe.unet,
    unet_0=pipe.unet,
    p_e=p_e,
    p_g=p_o0,
    m_excl=0.0,
    m_incl=0.0,
    layer_filter="attn2",
    # sim_param_group="avg_token"
    
    sim_param_group="attn_head"
)

tensor([[0.5898, 0.6094, 0.5234, 0.7656, 0.2832, 0.8047, 0.7930, 0.8945]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.4199, 0.8242, 0.5312, 0.8906, 0.6445, 0.8672, 0.7852, 0.8789]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.7539, 0.5586, 0.7656, 0.6992, 0.7539, 0.7812, 0.8906, 0.7891]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.8984, 0.8828, 0.8359, 0.7773, 0.8750, 0.9375, 0.9102, 0.8906]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.6875, 0.8477, 0.4434, 0.6172, 0.3984, 0.4297, 0.6094, 0.5547]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.7578, 0.9180, 0.6562, 0.7188, 0.5273, 0.5156, 0.6758, 0.5781]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<SumBackward1>)
tensor([[0.5508, 0.8047, 0.5859, 0.5352, 0.7422, 0.6602, 0.6914, 0.5469]],
       device='cuda:3', dtype=torch.bfloat16, grad_fn=<

(tensor(1., device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(0., device='cuda:3', dtype=torch.bfloat16, grad_fn=<MeanBackward0>),
 tensor(1., device='cuda:3', dtype=torch.bfloat16, grad_fn=<AddBackward0>))

In [66]:
 t1 = torch.tensor([[1,2,3],[4,5,6]]).float()
 t2 = torch.tensor([[1,2,3],[0,1,0]]).float()
 # cosine per head: [B, H]
 cos_excl_h = F.cosine_similarity(t1, t2, dim=-1)
 cos_excl_h

tensor([1.0000, 0.5698])

In [ ]:
# def compute_angular_exclusion_inclusion_loss(
#     unet_u,         # unlearned UNet (trainable)
#     unet_0,         # frozen reference UNet
#     p_e,            # erased prompt embedding [B, T, 768]
#     p_g,            # generic prompt embedding [B, T, 768]
#     m_excl: float,
#     m_incl: float,
#     layer_filter="attn2",  # cross-attention in diffusers SD1.4
#     use_bias: bool = False,
#     sim_param_group: str = "avg_token",  # {'avg_token', 'token', 'attn_head'}
# ):
#     """
#     Angular exclusion + inclusion loss in KV projection space (unsquared hinge).

#     sim_param_group:
#       - 'avg_token' : mean over tokens -> cosine -> hinge
#       - 'token'     : cosine per token -> hinge per token -> mean
#       - 'attn_head' : mean over tokens -> reshape to (num_heads=8) -> cosine per head
#                      -> hinge per head -> mean
#     """

#     assert sim_param_group in {"avg_token", "token", "attn_head"}

#     params_u = dict(unet_u.named_parameters())
#     params_0 = dict(unet_0.named_parameters())

#     excl_terms = []
#     incl_terms = []
#     matched_layers = 0

#     for name, W_u in params_u.items():
#         if layer_filter not in name:
#             continue
#         if not (name.endswith("to_k.weight") or name.endswith("to_v.weight")):
#             continue
#         if name not in params_0:
#             continue
#         # print(name)
#         # Frozen reference
#         W_0 = params_0[name].detach()

#         # Bias handling
#         if use_bias:
#             b_name = name.replace(".weight", ".bias")
#             b_u = params_u.get(b_name, None)
#             b_0 = params_0.get(b_name, None)
#             if b_0 is not None:
#                 b_0 = b_0.detach()
#         else:
#             b_u = None
#             b_0 = None

#         # Linear projections: [B, T, D]
#         W_u_e = F.linear(p_e, W_u, b_u)
#         with torch.no_grad():
#             W_0_e = F.linear(p_e, W_0, b_0)
#             W_0_g = F.linear(p_g, W_0, b_0)

#         # --------------------------------------------------
#         # Similarity + hinge computation modes
#         # --------------------------------------------------

#         if sim_param_group == "avg_token":
#             # [B, T, D] -> [B, D]
#             W_u_e_m = W_u_e.mean(dim=1)
#             W_0_e_m = W_0_e.mean(dim=1)
#             W_0_g_m = W_0_g.mean(dim=1)

#             cos_excl = F.cosine_similarity(W_u_e_m, W_0_e_m, dim=-1)  # [B]
#             cos_incl = F.cosine_similarity(W_u_e_m, W_0_g_m, dim=-1)  # [B]

#             # hinge on scalar cosine
#             excl_terms.append(torch.clamp(cos_excl - m_excl, min=0.0).mean())
#             incl_terms.append(torch.clamp(m_incl - cos_incl, min=0.0).mean())

#         elif sim_param_group == "token":
#             # cosine per token: [B, T]
#             cos_excl_tok = F.cosine_similarity(W_u_e, W_0_e, dim=-1)
#             cos_incl_tok = F.cosine_similarity(W_u_e, W_0_g, dim=-1)

#             # hinge per token, then mean over (B,T)
#             excl_terms.append(torch.clamp(cos_excl_tok - m_excl, min=0.0).mean())
#             incl_terms.append(torch.clamp(m_incl - cos_incl_tok, min=0.0).mean())

#         elif sim_param_group == "attn_head":
#             # mean over tokens first: [B, D]
#             W_u_e_m = W_u_e.mean(dim=1)
#             W_0_e_m = W_0_e.mean(dim=1)
#             W_0_g_m = W_0_g.mean(dim=1)

#             B, D = W_u_e_m.shape
#             num_heads = 8
#             assert D % num_heads == 0, "Feature dim must be divisible by num_heads"
#             head_dim = D // num_heads

#             # [B, D] -> [B, H, Dh]
#             W_u_e_h = W_u_e_m.view(B, num_heads, head_dim)
#             W_0_e_h = W_0_e_m.view(B, num_heads, head_dim)
#             W_0_g_h = W_0_g_m.view(B, num_heads, head_dim)

#             # cosine per head: [B, H]
#             cos_excl_h = F.cosine_similarity(W_u_e_h, W_0_e_h, dim=-1)
#             cos_incl_h = F.cosine_similarity(W_u_e_h, W_0_g_h, dim=-1)
            
#             # print(cos_excl_h)
#             print(cos_incl_h)

#             # hinge per head, then mean over (B,H)
#             excl_terms.append(torch.clamp(cos_excl_h - m_excl, min=0.0).mean())
#             incl_terms.append(torch.clamp(m_incl - cos_incl_h, min=0.0).mean())

#         matched_layers += 1

#     if matched_layers == 0:
#         zero = torch.tensor(0.0, device=p_e.device)
#         return zero, zero, zero

#     L_excl = torch.stack(excl_terms).mean()
#     L_incl = torch.stack(incl_terms).mean()
#     L_ang = L_excl + L_incl

#     return L_excl, L_incl, L_ang
# # import torch
# # import torch.nn.functional as F


# # def compute_angular_exclusion_inclusion_loss(
# #     unet_u,         # unlearned UNet (trainable)
# #     unet_0,         # frozen reference UNet
# #     p_e,            # erased prompt embedding [B, T, 768]
# #     p_g,            # generic prompt embedding [B, T, 768]
# #     m_excl: float,
# #     m_incl: float,
# #     layer_filter="attn2",  # cross-attention in diffusers SD1.4
# #     use_bias: bool = False,
# #     sim_param_group: str = "avg_token",  # {'avg_token', 'token', 'attn_head'}
# # ):
# #     """
# #     Angular exclusion + inclusion loss in KV projection space (unsquared hinge).

# #     sim_param_group:
# #       - 'avg_token' : mean over tokens, then cosine
# #       - 'token'     : token-wise cosine, then mean
# #       - 'attn_head' : mean over tokens, reshape to (num_heads=8), cosine per head, then mean
# #     """

# #     assert sim_param_group in {"avg_token", "token", "attn_head"}

# #     params_u = dict(unet_u.named_parameters())
# #     params_0 = dict(unet_0.named_parameters())

# #     excl_terms = []
# #     incl_terms = []
# #     matched_layers = 0

# #     for name, W_u in params_u.items():
# #         if layer_filter not in name:
# #             continue
# #         if not (name.endswith("to_k.weight") or name.endswith("to_v.weight")):
# #             print("Skipping param:", name)
# #             continue
# #         if name not in params_0:
# #             continue

# #         # Frozen reference
# #         W_0 = params_0[name].detach()

# #         # Bias handling
# #         if use_bias:
# #             b_name = name.replace(".weight", ".bias")
# #             b_u = params_u.get(b_name, None)
# #             b_0 = params_0.get(b_name, None)
# #             if b_0 is not None:
# #                 b_0 = b_0.detach()
# #         else:
# #             b_u = None
# #             b_0 = None

# #         # Linear projections
# #         W_u_e = F.linear(p_e, W_u, b_u)  # [B, T, D]

# #         with torch.no_grad():
# #             W_0_e = F.linear(p_e, W_0, b_0)
# #             W_0_g = F.linear(p_g, W_0, b_0)

# #         # --------------------------------------------------
# #         # Similarity computation modes
# #         # --------------------------------------------------

# #         if sim_param_group == "avg_token":
# #             # [B, T, D] → [B, D]
# #             W_u_e_m = W_u_e.mean(dim=1)
# #             W_0_e_m = W_0_e.mean(dim=1)
# #             W_0_g_m = W_0_g.mean(dim=1)

# #             cos_excl = F.cosine_similarity(W_u_e_m, W_0_e_m, dim=-1)
# #             cos_incl = F.cosine_similarity(W_u_e_m, W_0_g_m, dim=-1)

# #         elif sim_param_group == "token":
# #             # cosine per token → mean over tokens
# #             cos_excl_tok = F.cosine_similarity(W_u_e, W_0_e, dim=-1)  # [B, T]
# #             cos_incl_tok = F.cosine_similarity(W_u_e, W_0_g, dim=-1)

# #             cos_excl = cos_excl_tok.mean(dim=1)
# #             cos_incl = cos_incl_tok.mean(dim=1)

# #         elif sim_param_group == "attn_head":
# #             # mean over tokens first
# #             W_u_e_m = W_u_e.mean(dim=1)  # [B, D]
# #             W_0_e_m = W_0_e.mean(dim=1)
# #             W_0_g_m = W_0_g.mean(dim=1)

# #             B, D = W_u_e_m.shape
# #             num_heads = 8
# #             assert D % num_heads == 0, "Feature dim must be divisible by num_heads"

# #             head_dim = D // num_heads

# #             # [B, D] → [B, H, Dh] 
# #             W_u_e_h = W_u_e_m.view(B, num_heads, head_dim)
# #             W_0_e_h = W_0_e_m.view(B, num_heads, head_dim)
# #             W_0_g_h = W_0_g_m.view(B, num_heads, head_dim)
# #             # print(W_u_e_h.shape, W_0_e_h.shape, W_0_g_h.shape) # [1, 8, 40]), [1, 8, 80], [1, 8, 160]

# #             # cosine per head → mean over heads
# #             cos_excl_h = F.cosine_similarity(W_u_e_h, W_0_e_h, dim=-1)  # [B, H]
# #             cos_incl_h = F.cosine_similarity(W_u_e_h, W_0_g_h, dim=-1)

# #             # print(cos_excl_h.shape, cos_incl_h.shape)  # [1, 8]
# #             cos_excl = cos_excl_h.mean(dim=1)
# #             cos_incl = cos_incl_h.mean(dim=1)
            
# #         # --------------------------------------------------
# #         # Unsquared hinge losses
# #         # --------------------------------------------------
# #         excl_terms.append(torch.clamp(cos_excl - m_excl, min=0.0).mean())
# #         incl_terms.append(torch.clamp(m_incl - cos_incl, min=0.0).mean())
# #         matched_layers += 1

# #     if matched_layers == 0:
# #         zero = torch.tensor(0.0, device=p_e.device)
# #         return zero, zero, zero

# #     L_excl = torch.stack(excl_terms).mean()
# #     L_incl = torch.stack(incl_terms).mean()
# #     L_ang = L_excl + L_incl

# #     return L_excl, L_incl, L_ang
